# 01. 원본 데이터 라벨링 (Stratus, Splunk, BOTS v3)

원본 CloudTrail 이벤트에 정상/공격, 기법, 전술 라벨을 붙인다. 규칙은 `labels/README.md`.

```
[이 노트북] → preprocess.ipynb → window.ipynb → 04_split_v2.ipynb
data/raw/ → data/interim/event_labels.csv
```

**라벨 열**

| 열 | 내용 |
|---|---|
| `is_attack` | 정상 / 공격 |
| `technique` | 이벤트의 기법 1개 (MITRE ID, 3절 규칙으로 고친 값) |
| `techniques` | 기법 + 그 상위 기법 (예: `T1098|T1098.001`) (L3) |
| `tactics` | 기법이 속한 MITRE 전술 각각 (예: `TA0003|TA0004`) (L1, L2) |
| `attack_id` | 기법 실행 1회 = 세션 + 기법 |
| `benign_type` | 정상 하위 유형 (분석용) |

규칙 표(Stratus 매핑, Splunk·BOTS v3 공격 규칙, 기법 고치기)는 이 노트북 안에 있다. `labels/` 아래 CSV는 git에서 무시되거나 팀이 지워서 공유되지 않기 때문이다.

## 0. 경로, v19 라벨 표

In [1]:
import csv, hashlib, io, json, re, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "interim" / "event_labels.csv"
V19_PATH = ROOT / "labels" / "attack_labels_v19.csv"
V19_COMMIT = "74782a0"      # 팀이 저장소에서 지우기 전 v19 표가 있는 커밋

if V19_PATH.exists():
    v19 = pd.read_csv(V19_PATH).set_index("node_id")
else:                        # 로컬 파일이 없으면 git 기록에서 읽는다
    text = subprocess.run(["git", "-C", str(ROOT), "show", f"{V19_COMMIT}:labels/attack_labels_v19.csv"],
                          capture_output=True, text=True, check=True).stdout
    v19 = pd.read_csv(io.StringIO(text)).set_index("node_id")


def tactics_of(t):
    """기법이 속한 MITRE 전술 (v19 상위 관계). 예: T1078.004 → {TA0001, TA0003, TA0004, TA0005}"""
    if v19.at[t, "level"] == "top":
        return {t}
    return set().union(*[tactics_of(p) for p in str(v19.at[t, "parent_ids"]).split("|")])


def with_parents(t):
    """기법과 그 상위 기법 (전술 제외). 예: T1098.001 → {T1098, T1098.001}"""
    if v19.at[t, "level"] == "top":
        return set()
    return {t}.union(*[with_parents(p) for p in str(v19.at[t, "parent_ids"]).split("|")])


print("v19 노드", len(v19), "| 예:", sorted(with_parents("T1098.001")), sorted(tactics_of("T1078.004")))

v19 노드 83 | 예: ['T1098', 'T1098.001'] ['TA0001', 'TA0003', 'TA0004', 'TA0005']


## 1. 기법 ID 정리 규칙 (README 3절)

- Splunk가 상위 기법(T1078)으로만 라벨했지만 이벤트로 하위 기법이 분명한 것은 하위 기법으로 고친다
- v19 표의 사용자 정의 ID(MITRE에 없음)는 상위 MITRE 기법으로 바꾼다

In [2]:
# (시나리오, API) → 기법
TECHNIQUE_FIX = {
    ("assume_role_with_saml", "AssumeRoleWithSAML"): "T1078.004",        # SAML로 클라우드 계정 로그인
    ("aws_createloginprofile", "ConsoleLogin"): "T1078.004",             # 콘솔 로그인
    ("aws_createaccesskey", "CreateAccessKey"): "T1098.001",             # 다른 사용자에게 액세스 키 발급
    ("aws_createloginprofile", "CreateLoginProfile"): "T1098.001",       # 로그인 비밀번호 발급
    ("aws_updateloginprofile", "UpdateLoginProfile"): "T1098.001",       # 로그인 비밀번호 변경
    ("aws_create_policy_version", "CreatePolicyVersion"): "T1098.003",   # 정책 버전을 바꿔 권한 확대
    ("aws_setdefaultpolicyversion", "SetDefaultPolicyVersion"): "T1098.003",
}
NOT_MITRE = {"T1552.GEN": "T1552", "T1485.GEN": "T1485", "T1685.SEC": "T1685"}


def fix_technique(scenario, event_name, technique):
    if not isinstance(technique, str):
        return technique
    technique = TECHNIQUE_FIX.get((scenario, event_name), technique)
    return NOT_MITRE.get(technique, technique)

## 2. 공통 함수 (JSON 읽기, 이벤트 필드, 정상 하위 유형)

In [3]:
HELPER_CALLS = {"AssumeRole", "GetCallerIdentity"}


def read_json_events(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    if not text:
        return []
    try:
        data = json.loads(text)
        data = data if isinstance(data, list) else [data]
    except json.JSONDecodeError:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    # Splunk 검색 결과 형식: result._raw에 원본 CloudTrail이 있으면 꺼낸다
    return [json.loads(e["result"]["_raw"]) if "_raw" in (e.get("result") or {}) else e for e in data]


def event_row(e):
    ui = e.get("userIdentity") or {}
    return {"eventID": e.get("eventID"), "eventTime": e.get("eventTime"), "eventSource": e.get("eventSource"),
            "eventName": e.get("eventName"), "awsRegion": e.get("awsRegion"), "userType": ui.get("type"),
            "userArn": ui.get("arn") or ui.get("invokedBy"), "sourceIPAddress": e.get("sourceIPAddress"),
            "userAgent": e.get("userAgent"), "readOnly": e.get("readOnly"), "errorCode": e.get("errorCode")}


def benign_type(r):
    arn, ua = str(r["userArn"] or ""), str(r["userAgent"] or "").lower()
    if "AWSServiceRole" in arn or arn.endswith(".amazonaws.com") or r["userType"] == "AWSService":
        return "BEN.AUTO"
    if any(t in ua for t in ("terraform", "cloudformation", "cdk", "pulumi")):
        return "BEN.DEPLOY"
    if r["readOnly"] is True or str(r["eventName"]).startswith(("Describe", "List", "Get")):
        return "BEN.MONITOR"
    return "BEN.OPS"

## 3. Stratus (README 2-1)

파일 안 이벤트는 전부 공격. 기법은 파일명(Stratus ID)으로 정한다. 파일명의 전술은 쓰지 않는다.

In [4]:
STRATUS_MAPPING = {
    "aws.credential-access.ec2-get-password-data": "T1552.GEN",          # 1절 규칙으로 T1552가 됨
    "aws.discovery.ec2-download-user-data": "T1552.GEN",
    "aws.credential-access.ec2-steal-instance-credentials": "T1552.005",
    "aws.credential-access.secretsmanager-batch-retrieve-secrets": "T1555.006",
    "aws.credential-access.secretsmanager-retrieve-secrets": "T1555.006",
    "aws.credential-access.ssm-retrieve-securestring-parameters": "T1555.006",
    "aws.defense-evasion.cloudtrail-delete": "T1685.002",
    "aws.defense-evasion.cloudtrail-event-selectors": "T1685.002",
    "aws.defense-evasion.cloudtrail-stop": "T1685.002",
    "aws.defense-evasion.dns-delete-logs": "T1685.002",
    "aws.defense-evasion.vpc-remove-flow-logs": "T1685.002",
    "aws.defense-evasion.organizations-leave": "T1666",
    "aws.execution.ec2-launch-unusual-instances": "T1496.001",
    "aws.execution.ec2-user-data": "T1578.005",
    "aws.execution.ssm-send-command": "T1651",
    "aws.execution.ssm-start-session": "T1651",
    "aws.exfiltration.ec2-security-group-open-port-22-ingress": "T1686.001",
    "aws.exfiltration.ec2-share-ami": "T1537",
    "aws.exfiltration.ec2-share-ebs-snapshot": "T1537",
    "aws.exfiltration.ec2-share-ebs-volume": "T1537",
    "aws.exfiltration.rds-share-snapshot": "T1537",
    "aws.impact.bedrock-invoke-model": "T1496.004",
    "aws.initial-access.console-login-without-mfa": "T1078.004",
    "aws.lateral-movement.ec2-instance-connect": "T1021.008",
    "aws.lateral-movement.ec2-serial-console-send-ssh-public-key": "T1021.008",
    "aws.persistence.iam-backdoor-role": "T1098.003",
    "aws.persistence.iam-create-backdoor-role": "T1098.003",
    "aws.persistence.iam-backdoor-user": "T1098.001",
    "aws.persistence.iam-create-user-login-profile": "T1098.001",
    "aws.persistence.rolesanywhere-create-trust-anchor": "T1098.001",
    "aws.persistence.sts-federation-token": "T1098.001",
    "aws.privilege-escalation.iam-update-user-login-profile": "T1098.001",
    "aws.persistence.iam-create-admin-user": "T1136.003",
    "aws.persistence.lambda-backdoor-function": "T1546",
    "aws.persistence.lambda-layer-extension": "T1648",
    "aws.persistence.lambda-overwrite-code": "T1648",
}


def load_stratus():
    rows = []
    for path in sorted((RAW / "stratus").glob("*.json")):
        sid = path.stem
        for i, e in enumerate(read_json_events(path)):
            r = event_row(e)
            tech = fix_technique(sid, r["eventName"], STRATUS_MAPPING[sid])
            flags = [f for f, on in [("error_call", r["errorCode"]), ("helper_call", r["eventName"] in HELPER_CALLS)] if on]
            r.update(source="stratus", source_file=str(path.relative_to(ROOT)), scenario=sid, session=f"stratus:{sid}",
                     seq=i, attack_id=f"stratus:{sid}", is_attack=True, technique=tech,
                     label_source="stratus_file", flags="|".join(flags))
            rows.append(r)
    return rows


stratus_rows = load_stratus()
print("Stratus", len(stratus_rows), "events,", len({r["session"] for r in stratus_rows}), "files")

Stratus 311 events, 36 files


## 4. Splunk attack_data (README 2-2)

- 파일 선택: manifest `status`가 `있음`, `매핑(v19 ID 변경)`, `상위 기법만 있음`이고 CloudTrail 원본(JSON)인 것
- 배경 이벤트가 섞인 시나리오는 공격 API만 공격, 나머지는 정상. 그 외 시나리오는 파일 전체가 공격
- 내용이 같은 파일(aws_create_acl, aws_delete_acl)은 세션 하나로

In [5]:
SPLUNK_ATTACK_EVENTS = {
    "aws_createaccesskey": {"CreateAccessKey"},
    "aws_create_acl": {"CreateNetworkAcl", "CreateNetworkAclEntry", "ReplaceNetworkAclEntry"},
    "aws_delete_acl": {"DeleteNetworkAclEntry"},           # aws_create_acl과 같은 파일
    "aws_kms_key": {"CreateKey", "PutKeyPolicy"},
}
SPLUNK_USABLE_STATUS = {"있음", "매핑(v19 ID 변경)", "상위 기법만 있음"}


def load_splunk():
    base = RAW / "splunk_attack_data"
    manifest = pd.read_csv(base / "manifest_cloudtrail.csv", encoding="utf-8-sig").fillna("")
    rows, skipped, seen = [], [], {}
    for m in manifest.itertuples():
        reason = None
        if m.format != "CloudTrail 원본":
            reason = "OCSF"
        elif m.status not in SPLUNK_USABLE_STATUS or not m.our_label:
            reason = f"status={m.status}"
        elif not m.file.endswith(".json"):
            reason = "not json"
        if reason is None:
            events = read_json_events(base / m.file)
            if not events:
                reason = "empty"
            elif "result" in events[0] and "eventName" not in events[0]:
                reason = "splunk search export (not raw CloudTrail)"
        if reason:
            skipped.append((m.scenario, m.format, reason))
            continue
        attack_names = SPLUNK_ATTACK_EVENTS.get(m.scenario)
        digest = hashlib.sha1((base / m.file).read_bytes()).hexdigest()
        session = seen.setdefault(digest, f"splunk:{m.scenario}")
        for i, e in enumerate(events):
            r = event_row(e)
            is_attack = attack_names is None or r["eventName"] in attack_names
            tech = fix_technique(m.scenario, r["eventName"], m.our_label) if is_attack else None
            r.update(source="splunk", source_file=str((base / m.file).relative_to(ROOT)), scenario=m.scenario,
                     session=session, seq=i, attack_id=f"{session}:{tech}" if is_attack else None,
                     is_attack=is_attack, technique=tech,
                     label_source=("splunk_rule" if attack_names else "manifest") if is_attack else "splunk_background",
                     flags="error_call" if r["errorCode"] else "")
            rows.append(r)
    return rows, skipped


splunk_rows, skipped = load_splunk()
print("Splunk", len(splunk_rows), "events | 뺀 파일", len(skipped))
print(pd.DataFrame(skipped, columns=["scenario", "format", "reason"]).reason.value_counts().to_string())

Splunk 35031 events | 뺀 파일 44
reason
OCSF                                         27
splunk search export (not raw CloudTrail)     7
status=없음                                     7
not json                                      2
status=행위 기반(기법 폴더 아님)                        1


## 5. BOTS v3 (README 2-3)

계정 하나의 연속 기록이라 세션 하나. 위에서부터 처음 맞는 규칙을 쓴다. `EXCLUDE`는 데이터에서 뺀다 (공격인지 불확실).

In [6]:
# (주체, IP 또는 None, API 정규식, 기법)
BOTSV3_RULES = [
    ("web_admin", "139.198.18.205", r"^(RunInstances|CreateDefaultVpc)$", "T1496.001"),   # 유출 키로 15개 리전에 인스턴스 시도
    ("web_admin", "139.198.18.205", r"^(Describe|List|Get)(?!User$|AccessKeys$|CallerIdentity$|SessionToken$)", "T1580"),
    ("web_admin", "139.198.18.205", r".*", "T1078.004"),                                   # 그 외 유출 키 사용
    ("web_admin", None, r".*", "EXCLUDE"),           # 다른 IP의 web_admin: 공격자인지 내부 자동화인지 불확실
    ("bstoll", None, r"^PutBucketAcl$", "EXCLUDE"),  # 버킷 공개 후 되돌림: 공격인지 설정 실수인지 불확실
]


def load_botsv3():
    path = RAW / "botsv3" / "cloudtrail" / "botsv3_cloudtrail.json"
    if not path.exists():
        return [], 0
    rows, excluded = [], 0
    for i, e in enumerate(sorted(read_json_events(path), key=lambda e: e["eventTime"])):
        r = event_row(e)
        actor = (e.get("userIdentity") or {}).get("userName")
        tech = next((t for a, ip, pat, t in BOTSV3_RULES
                     if a == actor and (ip is None or ip == r["sourceIPAddress"]) and re.match(pat, r["eventName"] or "")), None)
        if tech == "EXCLUDE":
            excluded += 1
            continue
        is_attack = tech is not None
        r.update(source="botsv3", source_file=str(path.relative_to(ROOT)), scenario="frothly", session="botsv3:frothly",
                 seq=i, attack_id=f"botsv3:{actor}@{r['sourceIPAddress']}:{tech}" if is_attack else None,
                 is_attack=is_attack, technique=tech,
                 label_source="botsv3_rule" if is_attack else "botsv3_background",
                 flags="error_call" if r["errorCode"] else "")
        rows.append(r)
    return rows, excluded


bots_rows, bots_excluded = load_botsv3()
print("BOTS v3", len(bots_rows), "events | 뺀 이벤트", bots_excluded)

BOTS v3 6560 events | 뺀 이벤트 11


## 6. 합치기, 라벨 열 만들기, 저장

[볼 것] 모든 기법이 v19 표에 있고 MITRE에 없는 ID가 남지 않아야 함.

In [7]:
df = pd.DataFrame(stratus_rows + splunk_rows + bots_rows)
# 공유 파일의 중복 이벤트 정리: 공격 라벨이 붙은 쪽을 남긴다
df["event_uid"] = df["session"] + ":" + df["eventID"].fillna(df["seq"].astype(str))
df = (df.sort_values("is_attack", ascending=False).drop_duplicates("event_uid")
        .sort_values(["session", "eventTime", "seq"]).reset_index(drop=True))

techs = set(df["technique"].dropna())
assert not techs - set(v19.index), f"v19 표에 없는 기법: {techs - set(v19.index)}"
assert not techs & set(v19[v19.is_custom == "yes"].index), "MITRE에 없는 ID가 남음"

df["techniques"] = df["technique"].map(lambda t: "|".join(sorted(with_parents(t))) if isinstance(t, str) else "")
df["tactics"] = df["technique"].map(lambda t: "|".join(sorted(tactics_of(t))) if isinstance(t, str) else "")
df["label_level"] = df["technique"].map(lambda t: v19.at[t, "level"] if isinstance(t, str) else "")
df["benign_type"] = [None if a else benign_type(r) for a, (_, r) in zip(df["is_attack"], df.iterrows())]

OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"saved {OUT.relative_to(ROOT)}: {len(df)} events | 공격 {int(df.is_attack.sum())} | 정상 {int((~df.is_attack).sum())}")
print(df.groupby(["source", "is_attack"]).size().to_string())

saved data/interim/event_labels.csv: 28558 events | 공격 8695 | 정상 19863
source   is_attack
botsv3   False         5923
         True           637
splunk   False        13940
         True          7747
stratus  True           311


[볼 것] 기법별 라벨 (기법, 상위 포함 기법, 전술)

In [8]:
att = df[df.is_attack]
table = att.groupby("technique").agg(events=("event_uid", "size"), sources=("source", lambda s: ",".join(sorted(set(s)))),
                                     techniques=("techniques", "first"), tactics=("tactics", "first"))
table.insert(0, "name", [v19.at[t, "name"] for t in table.index])
table

,name,events,sources,techniques,tactics
technique,,,,,
T1021.008,Direct Cloud VM Connections,19,"splunk,stratus",T1021|T1021.008,TA0008
T1078.004,Cloud Accounts,79,"botsv3,splunk,stratus",T1078|T1078.004,TA0001|TA0003|TA0004|TA0005
T1087.004,Cloud Account,1,splunk,T1087|T1087.004,TA0007
T1098,Account Manipulation,245,splunk,T1098,TA0003|TA0004
T1098.001,Additional Cloud Credentials,67,"splunk,stratus",T1098|T1098.001,TA0003|TA0004
T1098.003,Additional Cloud Roles,11,"splunk,stratus",T1098|T1098.003,TA0003|TA0004
T1110,Brute Force,9,splunk,T1110,TA0006
T1110.003,Password Spraying,29,splunk,T1110|T1110.003,TA0006
T1136.003,Cloud Account,3,stratus,T1136|T1136.003,TA0003
